# Quantum Software Development

Author: Vanessa Knight
Full disclosure, I did use ChatGPT (Richard) to help me **organize** this Quantum Portfolio project.

## Objective

This notebook focuses on the software engineering skills required to build, analyze, optimize, and execute quantum programs.

Rather than introducing new algorithms, this notebook explains how professional quantum developers write maintainable, reusable, and efficient quantum software.

## Topics Covered

- Quantum Simulators
- Quantum Backends
- Transpilation
- Circuit Optimization
- Gate Decomposition
- Backend Properties
- Noise
- Resource Estimation
- Reusable Libraries
- Project Organization

## Learning Goals

By the end of this notebook, I should be able to:

- Execute circuits on simulators.
- Understand the role of a backend.
- Explain what transpilation does.
- Optimize circuits.
- Analyze circuit resources.
- Write reusable quantum code.
- Organize a professional quantum software project.

> This notebook brings together everything learned throughout the previous notebooks and focuses on practical quantum software engineering.

# 1. Simulators vs. Real Quantum Hardware

When developing quantum software, we rarely begin on a real quantum computer.

Instead, we usually develop using a **quantum simulator**.

A simulator behaves like an ideal quantum computer running on a classical computer.

Examples include:

- AerSimulator
- Statevector Simulator

Simulators are useful because they are:

- Fast
- Free
- Noise-free
- Easy to debug

Once a circuit behaves correctly, it can be executed on real quantum hardware.

## Backend

A **backend** is the system that executes a quantum circuit.

The backend may be:

- A simulator
- A real quantum computer

For example,

```python
backend = AerSimulator()
```

creates a simulator backend.

Later,

```python
backend = service.backend("...")
```

may refer to a real IBM Quantum computer.

The same quantum circuit can often be executed on different backends.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2,2)

qc.h(0)
qc.cx(0,1)

qc.measure([0,1],[0,1])

backend = AerSimulator()

job = backend.run(qc, shots=1000)

result = job.result()

counts = result.get_counts()

print(counts)

### Notes

One of the biggest insights while learning quantum software development was realizing that a **backend is not another quantum circuit.**

Instead, it is the **execution environment** for a quantum circuit.

Think of it like this:

```text
Python Code

↓

Quantum Circuit

↓

Backend

↓

Execution

↓

Measurement Results
```

Changing the backend changes **where** the circuit runs, not **what** the circuit is.

# 2. Transpilation

Quantum circuits are written using **logical gates** such as

- H
- X
- Z
- CX

However, real quantum computers cannot always execute these gates exactly as written.

Before a circuit can run on a backend, it must be **transpiled**.

Transpilation rewrites the circuit into an equivalent form that is compatible with the selected backend.

Although the circuit may look different after transpilation, it performs the same computation.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

backend = AerSimulator()

qc = QuantumCircuit(1)

# These gates cancel each other
qc.x(0)
qc.x(0)

qc.h(0)
qc.h(0)

qc.z(0)
qc.z(0)

print("Original Depth :", qc.depth())

optimized = transpile(
    qc,
    backend=backend,
    optimization_level=3
)

print("Optimized Depth:", optimized.depth())

print("\nOriginal Operations:")
print(qc.count_ops())

print("\nOptimized Operations:")
print(optimized.count_ops())

optimized.draw("mpl")

## Why Did the Circuit Become Smaller?

Notice that the circuit originally contained

```python
X
X
```

Applying two X gates is equivalent to doing nothing because

$$
X^2 = I
$$

Likewise,

$$
H^2 = I
$$

and

$$
Z^2 = I.
$$

The transpiler recognizes these identities and removes unnecessary gates.

The optimized circuit performs exactly the same computation while requiring fewer operations.

## Optimization Levels

Qiskit provides four optimization levels.

| Level | Description |
|--------|-------------|
| 0 | No optimization |
| 1 | Basic optimizations |
| 2 | More aggressive optimizations |
| 3 | Highest optimization level |

Higher optimization levels generally produce:

- fewer gates,
- shallower circuits,
- better performance on noisy hardware.

However, they may also require more compilation time.

### Notes

One important thing I learned is that **transpilation does not change the algorithm.**

It changes **how** the algorithm is implemented.

For example,

```text
Original Circuit

↓

Transpiler

↓

Equivalent Circuit

↓

Same Measurement Results
```

The purpose of transpilation is to make the circuit more efficient and compatible with the target backend.

# 3. Gate Decomposition

Not every quantum gate can be executed directly on real quantum hardware.

Many complex gates are **decomposed** into simpler gates that the hardware natively supports.

For example, the **Toffoli gate (CCX)** is a three-qubit gate that is usually implemented using a sequence of simpler one- and two-qubit gates.

This process is called **gate decomposition**.

The decomposition performs exactly the same computation while using the hardware's native gate set.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3)

# Toffoli (CCX) gate
qc.ccx(0, 1, 2)

print("Original Circuit:")
display(qc.draw("mpl"))

# Show the decomposed circuit
decomposed = qc.decompose()

print("Decomposed Circuit:")
display(decomposed.draw("mpl"))

## Why Does the CCX Become Many Gates?

When you decompose the circuit, you'll notice that the single CCX gate becomes a sequence of gates such as:

- H
- T
- T† (T-dagger)
- CX

This happens because most quantum hardware does **not** have a native CCX gate.

Instead, the transpiler rewrites it into a sequence of supported gates that produces the same overall transformation.

Although the decomposed circuit is longer, it is executable on real devices.

## Native Gate Sets

Every quantum backend supports a particular **native gate set**.

Examples of native gates may include:

- X
- SX
- RZ
- CX

When a circuit contains gates outside the native gate set, the transpiler automatically decomposes them into supported operations.

This allows the same logical circuit to run on different quantum hardware platforms.

### Notes

One of the biggest "aha!" moments while learning quantum software development was realizing that:

The circuit I write is **not necessarily** the circuit that runs on the hardware.

Instead,

```text
Logical Circuit

↓

Transpiler

↓

Gate Decomposition

↓

Native Gates

↓

Quantum Hardware
```

The algorithm remains the same, but its implementation changes to match the capabilities of the target backend.

# 4. Backend Properties

Not all quantum computers are the same.

Each backend has different hardware characteristics that affect how well a quantum circuit will perform.

Before choosing a backend, it is common to inspect its properties, including:

- Number of qubits
- Qubit connectivity
- Gate error rates
- Readout error
- Coherence times (T₁ and T₂)

These properties help determine whether a backend is suitable for a particular quantum circuit.

# 4. Backend Properties

Not all quantum computers are the same.

Each backend has different hardware characteristics that affect how well a quantum circuit will perform.

Before choosing a backend, it is common to inspect its properties, including:

- Number of qubits
- Qubit connectivity
- Gate error rates
- Readout error
- Coherence times (T₁ and T₂)

These properties help determine whether a backend is suitable for a particular quantum circuit.

## Important Backend Properties

### Number of Qubits

The backend must contain enough physical qubits to execute the circuit.

For example,

- A Bell State requires at least **2 qubits**.
- Quantum Teleportation requires at least **3 qubits**.

### Connectivity

Not every qubit can interact directly with every other qubit.

Some quantum computers only allow certain pairs of qubits to perform two-qubit operations.

If two qubits are not directly connected, the transpiler inserts additional SWAP gates to move quantum information across the device.

These extra gates increase circuit depth and can introduce additional noise.

## Error Rates

Real quantum hardware is imperfect.

Every operation has a small probability of producing an incorrect result.

Common sources of error include:

- Single-qubit gate errors
- Two-qubit gate errors
- Measurement (readout) errors

Two-qubit gates, such as the CNOT gate, generally have higher error rates than single-qubit gates.

Because of this, quantum software engineers often try to reduce the number of two-qubit gates in a circuit.

## Coherence Times (T₁ and T₂)

Quantum information does not remain stable forever.

Two important measures describe how long a qubit remains useful.

### T₁ (Relaxation Time)

The average time before a qubit naturally relaxes from

$$
|1\rangle
$$

back to

$$
|0\rangle.
$$

### T₂ (Dephasing Time)

The average time over which a qubit maintains its quantum phase relationships.

Loss of phase information reduces interference and makes quantum algorithms less reliable.

Longer T₁ and T₂ times generally allow deeper and more accurate quantum circuits.

### Notes

Choosing a backend is more than selecting the one with the largest number of qubits.

A good backend should also have:

- Low gate error rates
- Long coherence times
- Suitable qubit connectivity
- Enough qubits for the circuit

In practice, a smaller device with lower error rates may outperform a larger but noisier device for a particular algorithm.

# 5. Noise and Error Mitigation

Real quantum computers are affected by noise.

Unlike a simulator, physical qubits interact with their environment, causing errors during computation.

Noise can affect:

- Quantum gates
- Measurements
- Stored quantum states

Instead of completely correcting errors (which requires fault-tolerant quantum computers), today's devices often use **error mitigation** techniques to improve results.

## Sources of Noise

Some common sources of noise include:

### Gate Errors

Every quantum gate has a small probability of producing an incorrect operation.

Two-qubit gates are generally noisier than single-qubit gates.

### Decoherence

Qubits gradually lose their quantum information over time due to interactions with the environment.

This limits how deep a quantum circuit can be before the results become unreliable.

### Readout Errors

Even if the computation is correct, the measurement itself may report the wrong classical bit.

For example,

the true state may be

```text
|1⟩
```

but the hardware incorrectly reports

```text
0
```

# Error Mitigation

Error mitigation attempts to improve measurement results **without** fully correcting quantum errors.

Some common strategies include:

- Repeating the circuit many times (shots)
- Calibrating measurement errors
- Choosing qubits with lower error rates
- Reducing circuit depth through transpilation
- Minimizing the number of two-qubit gates

Unlike quantum error correction, these techniques do not eliminate errors.

Instead, they reduce their impact on the final result.

## Error Mitigation vs. Error Correction

| Error Mitigation | Error Correction |
|------------------|------------------|
| Uses today's hardware | Requires fault-tolerant hardware |
| Reduces the impact of errors | Detects and corrects errors |
| Software-based techniques | Additional logical and physical qubits |
| Practical on NISQ devices | Future large-scale quantum computers |

Today's quantum software primarily relies on **error mitigation**, while large-scale fault-tolerant systems will rely on **quantum error correction**.

### Notes

One of the biggest realizations while learning quantum software development was that writing a correct algorithm is only part of the challenge.

On real hardware, engineers must also think about:

- How noisy is the device?
- Can the circuit be made shallower?
- Can expensive two-qubit gates be reduced?
- Which backend will produce the most reliable results?

In practice, improving a quantum program often means improving **both** the algorithm and its implementation.

# 6. Building Reusable Quantum Libraries

As projects grow, copying and pasting code between notebooks becomes difficult to maintain.

Instead, common quantum circuits should be organized into reusable functions.

Benefits include:

- Cleaner notebooks
- Less duplicated code
- Easier testing
- Easier debugging
- Reusable algorithms across multiple projects

This is standard software engineering practice.

## Example Project Structure

A professional quantum computing project might be organized as:

```text
quantum-portfolio/

│
├── notebooks/
│   ├── 01_Foundations.ipynb
│   ├── 02_Entanglement.ipynb
│   ├── 03_Grover.ipynb
│   ├── 04_QFT.ipynb
│   ├── 05_VQE.ipynb
│   ├── 06_Shor.ipynb
│   └── 07_Quantum_Software_Development.ipynb
│
├── algorithms.py
├── utils.py
├── requirements.txt
├── README.md
└── LICENSE
```

## algorithms.py

The purpose of `algorithms.py` is to store reusable implementations of quantum algorithms.

Examples include:

```python
bell_state()

teleportation()

grover()

qft()

vqe_ansatz()

shor_demo()
```

Each function should focus on building or executing a single algorithm.

This allows the same implementation to be reused in notebooks, scripts, and larger applications.

## utils.py

The `utils.py` file stores helper functions that are useful across multiple projects.

Examples include:

- Executing circuits
- Printing resource metrics
- Drawing circuits
- Running simulations
- Plotting measurement results

Separating utility functions from algorithms keeps the codebase organized and easier to maintain.

## Example

Instead of repeatedly writing

```python
backend = AerSimulator()

job = backend.run(qc, shots=1000)

result = job.result()

counts = result.get_counts()

print(counts)
```

you could create a helper function:

```python
def run_circuit(qc, shots=1000):
    backend = AerSimulator()
    job = backend.run(qc, shots=shots)
    return job.result().get_counts()
```

Then simply write

```python
counts = run_circuit(qc)
```

This makes your code shorter, easier to read, and easier to maintain.

### Notes

One of the biggest transitions from learning quantum computing to becoming a quantum software developer is learning to think in terms of reusable components.

Instead of asking:

> "How do I solve this one problem?"

software engineers ask:

> "How can I write this once and reuse it everywhere?"

Applying software engineering principles such as modular design, reusable functions, and clear project organization makes quantum applications easier to develop, test, and extend.

# Final Portfolio Summary

Congratulations!

You have completed a complete introduction to quantum computing and quantum software development.

Throughout this portfolio, you progressed from the fundamentals of qubits to implementing some of the most important quantum algorithms and learning how professional quantum software is developed.

## Completed Topics

### Notebook 1

- Quantum Circuits
- Qubits
- Hilbert Space
- Quantum Gates
- Superposition
- Bloch Sphere
- Statevectors
- Rotation Gates

### Notebook 2

- Multi-Qubit Systems
- Tensor Products
- CNOT Gates
- Bell States
- Entanglement
- Quantum Teleportation

### Notebook 3

- Grover's Search Algorithm
- Oracle
- Diffusion Operator
- Amplitude Amplification

### Notebook 4

- Quantum Fourier Transform
- Controlled Phase Gates
- Phase Information

### Notebook 5

- Variational Quantum Eigensolver
- Hybrid Quantum-Classical Computing
- Ansatz
- Classical Optimization

### Notebook 6

- Shor's Algorithm
- Period Finding
- Modular Arithmetic
- Quantum Fourier Transform
- Classical Post-Processing

### Notebook 7

- Simulators
- Backends
- Transpilation
- Circuit Optimization
- Gate Decomposition
- Backend Properties
- Noise
- Error Mitigation
- Resource Analysis
- Software Organization

This portfolio provides a strong foundation for further study in quantum computing and quantum software engineering.

# 🎯 Final Interview Checklist

Before applying for quantum software roles, make sure you can confidently explain:

## Foundations

☐ What is a qubit?

☐ What is superposition?

☐ What is measurement?

☐ What is a Statevector?

☐ What is the Bloch Sphere?

## Multi-Qubit Systems

☐ Tensor products

☐ CNOT gate

☐ Bell States

☐ Entanglement

☐ Quantum Teleportation

## Algorithms

☐ Grover's Algorithm

☐ Quantum Fourier Transform

☐ Variational Quantum Eigensolver (VQE)

☐ Shor's Algorithm

## Quantum Software Development

☐ Simulators vs. Hardware

☐ Transpilation

☐ Optimization Levels

☐ Backend Properties

☐ Noise

☐ Error Mitigation

☐ Circuit Depth

☐ Circuit Size

☐ Gate Counts

☐ Gate Decomposition

## Software Engineering

☐ Reusable functions

☐ Modular project structure

☐ Resource estimation

☐ Writing maintainable quantum software

# Where to Go Next

With the foundations complete, possible next areas of study include:

## Quantum Machine Learning (QML)

- Quantum Neural Networks
- Quantum Kernels
- Data Encoding
- Hybrid Learning Models

## Quantum Chemistry

- Molecular Hamiltonians
- Electronic Structure Problems
- Larger VQE Applications

## Quantum Error Correction

- Logical Qubits
- Surface Codes
- Stabilizer Codes

## Advanced Quantum Algorithms

- Quantum Phase Estimation
- HHL Algorithm
- Quantum Walks
- Amplitude Estimation

## Quantum Hardware

- Superconducting Qubits
- Trapped Ions
- Neutral Atoms
- Photonic Quantum Computing

# Final Thoughts

Quantum computing combines ideas from computer science, mathematics, and physics.

Learning the subject takes time because each topic builds upon the previous ones.

This portfolio represents the completion of a structured journey through:

- Quantum computing fundamentals
- Core quantum algorithms
- Practical quantum software engineering

The next step is to continue building projects, experimenting with real quantum hardware, and exploring advanced topics.

Learning quantum computing is an ongoing process, and this portfolio serves as a strong foundation for that journey.